# Auto Generated Agent Chat: Group Chat with GPTAssistantAgent

````{=mdx}
:::warning
`GPTAssistantAgent` is deprecated as of v0.12 and will be removed in v0.14. Use `ConversableAgent` instead. This notebook will also be removed in v0.14.
:::
````

AG2 offers conversable agents powered by LLM, tool or human, which can be used to perform tasks collectively via automated chat. This framework allows tool use and human participation through multi-agent conversation.
Please find documentation about this feature [here](https://docs.ag2.ai/latest/docs/user-guide/basic-concepts/conversable-agent/).

In this notebook, we demonstrate how to get multiple `GPTAssistantAgent` converse through group chat.

## Requirements

AG2 requires `Python>=3.9`. To run this notebook example, please install:
````{=mdx}
:::info Requirements
Install `autogen`:
```bash
pip install autogen
```

For more information, please refer to the [installation guide](https://docs.ag2.ai/latest/docs/user-guide/basic-concepts/installing-ag2).
:::
````

## Set your API Endpoint

The [`LLMConfig.from_json`](https://docs.ag2.ai/latest/docs/api-reference/autogen/llm_config/LLMConfig) method loads a list of configurations from an environment variable or a json file.

In [ ]:
import os

import autogen
from autogen.agentchat import AssistantAgent
from autogen.agentchat.contrib.gpt_assistant_agent import GPTAssistantAgent

llm_config = autogen.LLMConfig(
    config_list=[
        {
            "model": "gpt-5",
            "api_key": os.getenv("OPENAI_API_KEY"),
            "api_type": "openai",
        },
        {
            "model": "gpt-5-mini",
            "api_key": os.getenv("OPENAI_API_KEY"),
            "api_type": "openai",
        },
    ]
)

````{=mdx}
:::tip
Learn more about configuring LLMs for agents [here](https://docs.ag2.ai/latest/docs/user-guide/basic-concepts/llm-configuration).
:::
````

## Define GPTAssistantAgent and GroupChat

In [ ]:
# Define user proxy agent
user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin.",
    code_execution_config={
        "last_n_messages": 2,
        "work_dir": "groupchat",
        "use_docker": False,
    },  # Please set use_docker=True if docker is available to run the generated code. Using docker is safer than running the generated code directly.
    human_input_mode="TERMINATE",
)

# define two GPTAssistants
coder = GPTAssistantAgent(
    name="Coder",
    llm_config=llm_config,
    instructions=AssistantAgent.DEFAULT_SYSTEM_MESSAGE,
)

analyst = GPTAssistantAgent(
    name="Data_analyst",
    instructions="You are a data analyst that offers insight into data.",
    llm_config=llm_config,
)
# define group chat
groupchat = autogen.GroupChat(agents=[user_proxy, coder, analyst], messages=[], max_round=10)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

## Initiate Group Chat
Now all is set, we can initiate group chat.

In [ ]:
user_proxy.initiate_chat(
    manager,
    message="Get the number of issues and pull requests for the repository 'ag2ai/ag2' over the past three weeks and offer analysis to the data. You should print the data in csv format grouped by weeks.",
)
# type exit to terminate the chat